In [ ]:
import pandas as pd
import re
import matplotlib.pyplot as plt
import os

RED = "#BD8E00"
RED_LIGHT = '#FFE187'
RED_DARK = "#AF7B00"

BLUE = '#305497'
BLUE_LIGHT = '#9AAFD4'
BLUE_DARK = '#305497'

GRAY = '#404040'

file_path = 'ETTm2_robustness.txt'
if not os.path.exists(file_path):
    file_path = os.path.join('output', 'robustness', 'ETTm2_robustness.txt')

with open(file_path, 'r') as f:
    all_lines = f.readlines()

data = []
current_config = {}

for line in all_lines:
    line = line.strip()
    if not line:
        continue

    if line.startswith('ETTm2_'):
        pl_match = re.search(r'\bpl:(\d+)', line)
        nl_match = re.search(r'\bnl:([\d.]+)', line)
        pl = int(pl_match.group(1)) if pl_match else None
        nl = float(nl_match.group(1)) if nl_match else 0.0

        current_config = {'pl': pl, 'nl': nl}

    elif line.startswith('mse:'):
        parts = line.split(',')
        mse = float(parts[0].split(':')[1])
        mae = float(parts[1].split(':')[1])

        entry = current_config.copy()
        entry['mse'] = mse
        entry['mae'] = mae
        data.append(entry)

df = pd.DataFrame(data)
df = df.sort_values(by=['pl', 'nl']).reset_index(drop=True)
print("Raw data:")
print(df.to_string(index=False))

metrics = ['mse', 'mae']
noise_levels = [0.1, 0.2, 0.3, 0.5]
base_noise = 0.0

results = []
prediction_lengths = sorted(df['pl'].unique())

for pl in prediction_lengths:
    row = {'Prediction Length': pl}

    base_data = df[(df['pl'] == pl) & (df['nl'] == base_noise)]
    if base_data.empty:
        continue

    base_mse = base_data['mse'].values[0]
    base_mae = base_data['mae'].values[0]

    row[f'MSE (nl={base_noise})'] = round(base_mse, 3)
    row[f'MAE (nl={base_noise})'] = round(base_mae, 3)

    for nl in noise_levels:
        curr_data = df[(df['pl'] == pl) & (df['nl'] == nl)]
        if curr_data.empty:
            continue

        curr_mse = curr_data['mse'].values[0]
        curr_mae = curr_data['mae'].values[0]

        row[f'MSE (nl={nl})'] = round(curr_mse, 3)
        row[f'MAE (nl={nl})'] = round(curr_mae, 3)

        mse_pct_change = ((curr_mse - base_mse) / base_mse) * 100
        mae_pct_change = ((curr_mae - base_mae) / base_mae) * 100

        row[f'MSE % Change (nl={nl})'] = round(mse_pct_change, 3)
        row[f'MAE % Change (nl={nl})'] = round(mae_pct_change, 3)

    results.append(row)

results_df = pd.DataFrame(results)
results_df.set_index('Prediction Length', inplace=True)

print('\nResults with % change:')
display(results_df)

In [ ]:
import numpy as np

prediction_lengths = sorted(df['pl'].unique())
n_pl = len(prediction_lengths)
all_noise_levels = [0.0, 0.1, 0.2, 0.3, 0.5]

fig, axes = plt.subplots(1, n_pl, figsize=(6 * n_pl, 5), sharey=False)
if n_pl == 1:
    axes = [axes]

for i, pl in enumerate(prediction_lengths):
    ax = axes[i]

    mses = []
    maes = []
    valid_levels = []
    mse_changes = []
    mae_changes = []

    base_row = df[(df['pl'] == pl) & (df['nl'] == 0.0)]
    base_mse = base_row['mse'].values[0] if not base_row.empty else None
    base_mae = base_row['mae'].values[0] if not base_row.empty else None

    for nl in all_noise_levels:
        row = df[(df['pl'] == pl) & (df['nl'] == nl)]
        if not row.empty:
            curr_mse = row['mse'].values[0]
            curr_mae = row['mae'].values[0]
            mses.append(curr_mse)
            maes.append(curr_mae)
            valid_levels.append(nl)

            if base_mse:
                mse_changes.append(((curr_mse - base_mse) / base_mse) * 100)
            else:
                mse_changes.append(0)

            if base_mae:
                mae_changes.append(((curr_mae - base_mae) / base_mae) * 100)
            else:
                mae_changes.append(0)

    x = np.arange(len(valid_levels))
    width = 0.25

    rects1 = ax.bar(x - width / 2, mses, width, label='MSE', color=RED_LIGHT, edgecolor=RED, linewidth=3)
    rects2 = ax.bar(x + width / 2, maes, width, label='MAE', color=BLUE_LIGHT, edgecolor=BLUE, linewidth=3)

    all_vals = [v for v in mses + maes if v > 0]
    if all_vals:
        y_min = min(all_vals)
        y_max = max(all_vals)
        ax.set_ylim(y_min * 0.85, y_max * 1.15)

    ax.set_title(f'Prediction Length: {pl}', fontsize=16, fontweight='bold', color=GRAY)
    ax.set_xlabel('Std Dev', fontsize=15, fontweight='bold', color=GRAY)
    if i == 0:
        ax.set_ylabel('Error Value', fontsize=15, fontweight='bold', color=GRAY)
    ax.set_xticks(x)
    ax.set_xticklabels(valid_levels, fontsize=13, fontweight='bold')
    ax.tick_params(axis='y', labelsize=13, colors=GRAY)
    ax.tick_params(axis='x', colors=GRAY)
    for spine in ax.spines.values():
        spine.set_edgecolor(GRAY)
        spine.set_linewidth(1.5)

    ax2 = ax.twinx()
    line1 = ax2.plot(x, mse_changes, color=RED_DARK, marker='*', markersize=14, linestyle='-', linewidth=2, label='MSE % Change')
    line2 = ax2.plot(x, mae_changes, color=BLUE_DARK, marker='s', markersize=10, linestyle='-', linewidth=2, label='MAE % Change')

    if i == n_pl - 1:
        ax2.set_ylabel('Change Rate (%)', fontsize=15, fontweight='bold', color=GRAY)
    ax2.tick_params(axis='y', labelsize=13, colors=GRAY)
    for label in ax2.get_yticklabels():
        label.set_fontweight('bold')
    for sp in ax2.spines.values():
        sp.set_color(GRAY)
        sp.set_linewidth(0.8)

    y2_lo, y2_hi = ax2.get_ylim()
    y2_lo = 0
    ax2.set_ylim(y2_lo, y2_hi)
    ax2.set_yticks(np.linspace(y2_lo, int(np.ceil(y2_hi)), 6))
    ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{int(round(v))}'))

    y_lo, y_hi = ax.get_ylim()
    ax.set_yticks(np.linspace(y_lo, y_hi, 5))
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:.2f}'))
    for label in ax.get_yticklabels():
        label.set_fontweight('bold')

    lines = [rects1, rects2] + line1 + line2
    labels = [l.get_label() for l in lines]
    ax.legend(lines, labels, loc='upper left', fontsize=14)

    ax.grid(True, axis='y', linestyle='--', alpha=0.3, color=GRAY)

plt.tight_layout()
# plt.savefig('robustness_bar.png', bbox_inches='tight', dpi=300)
plt.show()